# HR Policy Question Answering System (RAG)
This notebook is an exploratory example of a retrieval-augmented question-answering pipeline over HR policy PDFs supplied by the user. The source PDFs are not included in this repository.

The workflow covers PDF ingestion, text cleanup, chunking, embeddings, a persisted Chroma vector store, and example questions. Context-only prompting is an instruction to the model, not a guarantee against hallucinations or prompt injection.

- Environment: Kaggle by default; other environments can set `HR_POLICY_PDF_DIR`
- Domain: Human Resources policies


## 1. Environment Setup

Install only the required libraries for a lightweight, clean RAG pipeline.

In [ ]:
!pip install -q -U pip
!pip install -q -U langchain langchain-community langchain-core langchain-huggingface langchain-chroma langchain-text-splitters 
!pip install -q -U pypdf chromadb sentence-transformers tqdm bitsandbytes accelerate transformers

## 2. Dataset Loading & Verification

Load HR Policy PDFs and check availability.

In [ ]:
import os

DATA_PATH = os.environ.get("HR_POLICY_PDF_DIR", "/kaggle/input/hr-policy-docs-pdf")
if not os.path.isdir(DATA_PATH):
    raise FileNotFoundError(
        f"PDF directory not found: {DATA_PATH}. Set HR_POLICY_PDF_DIR to a directory containing policy PDFs."
    )
pdf_files = sorted(
    filename for filename in os.listdir(DATA_PATH)
    if filename.lower().endswith(".pdf") and os.path.isfile(os.path.join(DATA_PATH, filename))
)
if not pdf_files:
    raise FileNotFoundError(f"No PDF files found in {DATA_PATH}.")
print("PDF files:", pdf_files)


## 3. PDF Document Ingestion & Text Cleaning & Normalization

Load each PDF page-by-page to preserve metadata (file name and page number).

Clean text from headers, footers, page numbers, bullets, and unnecessary whitespace.

In [ ]:
import re
from tqdm import tqdm
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document 

raw_documents = []
for file in tqdm(pdf_files, desc="Loading PDFs"):
    loader = PyPDFLoader(os.path.join(DATA_PATH, file))
    raw_documents.extend(loader.load())

def clean_text(text: str) -> str:
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"Page\s*\d+", "", text, flags=re.IGNORECASE)
    text = re.sub(r"HR Policy", "", text, flags=re.IGNORECASE)
    text = re.sub(r"[•●■□▪]", "", text)
    return text.strip()

cleaned_documents = []
for doc in tqdm(raw_documents, desc="Cleaning text"):
    text = clean_text(doc.page_content)
    if len(text) > 50:
        cleaned_documents.append(Document(page_content=text, metadata=doc.metadata))

print(f"Cleaned documents: {len(cleaned_documents)}")

## 4. Chunking &  Quality Control

Split documents into semantic chunks for better retrieval.

Filter weak or noisy chunks and clean metadata.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=120,
    separators=["\n\n", "\n", ".", " ", ""]
)

all_chunks = splitter.split_documents(cleaned_documents)

final_chunks = []
for chunk in all_chunks:
    if len(chunk.page_content) > 100:
        source_name = os.path.basename(chunk.metadata.get("source", "Unknown"))
        chunk.metadata["source"] = source_name
        final_chunks.append(chunk)

print(f"Final Chunks for Vector DB: {len(final_chunks)}")

# 5. Embeddings &  Vector Store (ChromaDB)

Convert chunks into vector embeddings.

Store embeddings with metadata for efficient semantic search.

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

VECTOR_DB_PATH = "chroma_hr_db"
vectorstore = Chroma.from_documents(
    documents=final_chunks,
    embedding=embeddings,
    persist_directory=VECTOR_DB_PATH
)

print("Vector Store is ready and persisted.")

# 6. LLM Setup (Llama-3 with 4-bit Quantization)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline
from langchain_huggingface import HuggingFacePipeline
if not torch.cuda.is_available():
    raise RuntimeError("The notebook's 4-bit bitsandbytes model requires a CUDA GPU.")
compute_dtype = (
    torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype
)
model_id = "unsloth/llama-3-8b-Instruct-bnb-4bit"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)
text_gen = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    do_sample=False,
    repetition_penalty=1.1,
    return_full_text=False
)
llm = HuggingFacePipeline(pipeline=text_gen)
print("Llama-3 is loaded and ready.")


# 7. RAG Chain & Anti-Hallucination Prompt

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

template = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are a professional HR Assistant. Use ONLY the following context to answer the question. 
If the answer is not in the context, strictly say: "I'm sorry, but this information is not available in the HR policies."

Context:
{context}<|eot_id|><|start_header_id|>user<|end_header_id|>

Question: {question}<|eot_id|><|start_header_id|>assistant<|end_header_id|>"""

prompt = PromptTemplate.from_template(template)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

def ask_hr(query):
    print(f"\n Question: {query}")
    
    response = rag_chain.invoke(query)
    
    docs = retriever.invoke(query)

    if "<|end_header_id|>" in response:
        clean_answer = response.split("<|end_header_id|>")[-1].strip()
    elif "assistant" in response:
        clean_answer = response.split("assistant")[-1].strip()
    else:
        clean_answer = response.strip()
    
    print(f" Answer: {clean_answer}")
    
    print("\n Sources :")
    sources = set([f"- {doc.metadata.get('source')} (Page: {doc.metadata.get('page', 'N/A')})" for doc in docs])
    for s in sources:
        print(s)

print("--- HR System Online (Clean Version) ---")
ask_hr("What is the policy for annual health checks?")
ask_hr("Can I bring a lion to the office?")

## 8. Example Queries (Manual Demonstrations)


In [ ]:
print("Start professional assessment tests...\n")

# 1. Semantic Search Test

print("--- Test 1: Indirect Query ---")
ask_hr("If I am feeling sick and want to get a medical check-up, how much will the company pay for me?")

# 2. Data Extraction & Comparison

print("\n--- Test 2: Comparison & Logic ---")
ask_hr("What is the difference in cost for a health check-up between a 30-year-old employee and a 40-year-old employee in grade E01?")

# 3. Boundary / Adversarial Test

print("\n--- Test 3: Out-of-Domain (Adversarial) ---")
ask_hr("Write a Python code to sum two numbers.")

# 4.Multi-source Retrieval

print("\n--- Test 4: Multi-context Search ---")
ask_hr("Summary of the rules for travel and the health check-up eligibility.")

## Conclusion
This notebook shows one example workflow for preparing a user-supplied PDF corpus and querying it with retrieval and a language model. The sample queries are manual demonstrations, not automated evaluation.

The results depend on the provided documents, retrieval, model, and runtime. This notebook does not establish answer accuracy, prevent hallucinations, or demonstrate production readiness. Verify any answer against the cited source policy.
